# 2B · Tarea B: extracción de entidades

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

Cada enfoque es una función `extraer(texto) -> lista de {"tipo", "texto", "valor"}`. FECHA, HORA, ASIGNATURA y PRIORIDAD necesitan `valor` normalizado; PERSONA y LUGAR, no.

In [ ]:
import time, json
from utilidades import cargar, guardar, particiones, INTENCIONES, TIPOS, HOY
from evaluar import evaluar, imprimir

train, val = particiones()
agenda = cargar("agenda.jsonl")
X_train, X_val = [x["texto"] for x in train], [x["texto"] for x in val]
RESULTADOS = {}      # enfoque -> métricas y predicciones en validación

def registrar(nombre, preds, segundos):
    r = evaluar(val, preds)
    rd = evaluar(val, preds, "dificil") if any(x["dificil"] for x in val) else {}   # hasta que tengáis frases difíciles
    RESULTADOS[nombre] = {"metrica": r["B_f1_micro"], "dificiles": rd.get("B_f1_micro"),
                          "ms_por_frase": 1000 * segundos / max(len(preds), 1), "preds": preds}
    print(f"{nombre:<40} B_f1_micro = {r['B_f1_micro']:.3f}   difíciles = {rd.get('B_f1_micro', '—')}")

In [ ]:
def preds_B(listas):
    return [{"id": x["id"], "entidades": e} for x, e in zip(val, listas)]

## Normalización
Detectar el trozo y normalizar su valor son dos problemas distintos. Estas funciones las comparten todos los enfoques que solo encuentran el trozo (B1, B2 y B3).

In [ ]:
def norm_fecha(trozo):        # «el jueves» -> "2026-02-12"; «esta semana» -> "2026-02-09/2026-02-15"
    return None               # TODO (usad HOY)
def norm_hora(trozo, frase=""):  # «a las 5» -> "17:00"; «cena a las 9» -> "21:00"
    return None               # TODO
def norm_asignatura(trozo):   # «BBDD» -> "Bases de Datos"
    return None               # TODO
def norm_prioridad(trozo):    # «urgente» -> "alta"; «sin prisa» -> "baja"
    return None               # TODO

# Comprobación: ¿cuántos valores aciertan vuestras funciones con los trozos de oro?
F = {"FECHA": norm_fecha, "ASIGNATURA": norm_asignatura, "PRIORIDAD": norm_prioridad}
ok = tot = 0
for x in train:
    for e in x["entidades"]:
        if "valor" in e:
            v = norm_hora(e["texto"], x["texto"]) if e["tipo"] == "HORA" else F[e["tipo"]](e["texto"])
            tot += 1; ok += v == e["valor"]
print(f"Normalización con los trozos de oro: {ok}/{tot}")

### B1 · Reglas y expresiones regulares
Listas (asignaturas, parentescos) y patrones de fecha y hora (temas 2 y 5).

In [ ]:
def extraer_B1(texto):
    return []          # TODO
t0 = time.time(); registrar("B1 Reglas y expresiones regulares", preds_B([extraer_B1(t) for t in X_val]), time.time() - t0)

### B2 · NER de spaCy + reglas
spaCy para PERSONA y LUGAR; reglas para el resto.

In [ ]:
def extraer_B2(texto):
    return []          # TODO
t0 = time.time(); registrar("B2 NER de spaCy + reglas", preds_B([extraer_B2(t) for t in X_val]), time.time() - t0)

### B3 · Embeddings de BETO sin ajustar + MLP
Un vector por subpalabra y un MLP que predice su etiqueta BIO.

In [ ]:
def extraer_B3(texto):
    return []          # TODO
t0 = time.time(); registrar("B3 Embeddings de BETO sin ajustar + MLP", preds_B([extraer_B3(t) for t in X_val]), time.time() - t0)

### B4 · LLM con salida JSON
Incluid en el prompt la fecha de hoy y las convenciones; validad el JSON.

In [ ]:
def extraer_B4(texto):
    return []          # TODO
t0 = time.time(); registrar("B4 LLM con salida JSON", preds_B([extraer_B4(t) for t in X_val]), time.time() - t0)

## Resumen y enfoque elegido
Guardamos los resultados de validación para el notebook 3 y elegimos un enfoque.

In [ ]:
import pandas as pd
print(pd.DataFrame({k: {c: v[c] for c in ("metrica", "dificiles", "ms_por_frase")} for k, v in RESULTADOS.items()}).T.round(3))
json.dump(RESULTADOS, open("resultados_B.json", "w", encoding="utf-8"), ensure_ascii=False)
MEJOR = max(RESULTADOS, key=lambda k: RESULTADOS[k]["metrica"])     # TODO: elegid y justificad el enfoque (calidad, coste, latencia…)
print("Enfoque elegido:", MEJOR)

## Exportar el enfoque elegido
Copiad a **`tarea_b.py`** el enfoque elegido (ya entrenado o con lo necesario para entrenarlo) como una función `extraer_entidades(texto) -> [{"tipo", "texto", "valor"}]`. La usan el asistente (notebook 4) y la función `generar_predicciones` del notebook 3, que es la que ejecutará el profesor con el **test oculto**.